In [1]:
# ============================================================
# TP Final - Siniestros Viales CABA
# Fase 4B - Modelado Avanzado Experimental
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.ensemble import RandomForestClassifier

from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier

RANDOM_STATE = 42

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
# ============================================================
# Carga de datos
# ============================================================

df = pd.read_csv("../data_processed/siniestros_viales_modelado.csv")

print("Dimensiones:", df.shape)

df.head()

Dimensiones: (65818, 17)


,id_siniestro,siniestro_fatal,fecha_siniestro,anio_siniestro,mes_siniestro,dia_siniestro,dia_semana,rango_horario,comuna_siniestro,tipo_de_via_siniestro,participantes_siniestro,modo_desplazamiento_victima,contraparte_siniestro,latitud_siniestro,longitud_siniestro,edad_promedio_victimas,sexo_victima_agregado
0,LC-2019-0000053,0,2019-01-01,2019,1,1,Tuesday,1.0,8.0,AUTOPISTA,AUTO-UTILITARIO,AUTO,UTILITARIO,-34.669125,-58.443510,57.000000,M
1,LC-2019-0000647,0,2019-01-01,2019,1,1,Tuesday,18.0,12.0,AVENIDA,MOTO-AUTO,MOTO,AUTO,-34.583403,-58.490436,54.000000,M
2,LC-2019-0000445,0,2019-01-01,2019,1,1,Tuesday,13.0,11.0,NaN,AUTO-AUTO,AUTO,AUTO,-34.603255,-58.524662,40.333333,MIXTO
3,LC-2019-0000194,0,2019-01-01,2019,1,1,Tuesday,7.0,9.0,NaN,AUTO-CAMION,AUTO,CAMION,-34.650156,-58.528413,33.000000,F
4,LC-2019-0000329,0,2019-01-01,2019,1,1,Tuesday,12.0,4.0,NaN,AUTO-MOVIL,AUTO,MOVIL,-34.648387,-58.404748,23.000000,M


In [3]:
# ============================================================
# Variables predictoras y objetivo
# ============================================================

X = df.drop(
    columns=[
        "id_siniestro",
        "fecha_siniestro",
        "siniestro_fatal"
    ]
).copy()

y = df["siniestro_fatal"]

# ============================================================
# Corrección de tipos detectada en Fase 4
# ============================================================

X["rango_horario"] = X["rango_horario"].astype(str)
X["comuna_siniestro"] = X["comuna_siniestro"].astype(str)

print("Shape X:", X.shape)
print("Shape y:", y.shape)

print("\nTipos corregidos:")
print(X[["rango_horario", "comuna_siniestro"]].dtypes)

Shape X: (65818, 14)
Shape y: (65818,)

Tipos corregidos:
rango_horario       object
comuna_siniestro    object
dtype: object


In [4]:
# ============================================================
# Variables numéricas y categóricas
# ============================================================

numericas = [
    "anio_siniestro",
    "mes_siniestro",
    "dia_siniestro",
    "latitud_siniestro",
    "longitud_siniestro",
    "edad_promedio_victimas"
]

categoricas = [
    col for col in X.columns
    if col not in numericas
]

print("Numéricas:", len(numericas))
print("Categóricas:", len(categoricas))

print("\nVariables numéricas:")
print(numericas)

print("\nVariables categóricas:")
print(categoricas)

Numéricas: 6
Categóricas: 8

Variables numéricas:
['anio_siniestro', 'mes_siniestro', 'dia_siniestro', 'latitud_siniestro', 'longitud_siniestro', 'edad_promedio_victimas']

Variables categóricas:
['dia_semana', 'rango_horario', 'comuna_siniestro', 'tipo_de_via_siniestro', 'participantes_siniestro', 'modo_desplazamiento_victima', 'contraparte_siniestro', 'sexo_victima_agregado']


In [5]:
# ============================================================
# Preprocesamiento
# ============================================================

transformador_numerico = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

transformador_categorico = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="SIN_DATO"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", transformador_numerico, numericas),
        ("cat", transformador_categorico, categoricas)
    ]
)

print("Preprocesador construido correctamente.")

Preprocesador construido correctamente.


In [6]:
# ============================================================
# División entrenamiento / prueba
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nDistribución train:")
print(y_train.value_counts(normalize=True))

print("\nDistribución test:")
print(y_test.value_counts(normalize=True))

Train: (52654, 14)
Test : (13164, 14)

Distribución train:
siniestro_fatal
0    0.989535
1    0.010465
Name: proportion, dtype: float64

Distribución test:
siniestro_fatal
0    0.989517
1    0.010483
Name: proportion, dtype: float64


In [7]:
# ============================================================
# Random Forest oficial - Fase 4
# ============================================================
# Antes de probar modelos más avanzados, se construye un Pipeline con Random Forest utilizando los mejores hiperparámetros encontrados en la Fase 3.

rf_oficial = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=None,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

print("Pipeline RF oficial creado.")

Pipeline RF oficial creado.


In [8]:
# ============================================================
# Entrenamiento y evaluación RF oficial
# ============================================================

rf_oficial.fit(X_train, y_train)

pred_rf = rf_oficial.predict(X_test)

accuracy_rf = accuracy_score(y_test, pred_rf)
precision_rf = precision_score(y_test, pred_rf)
recall_rf = recall_score(y_test, pred_rf)
f1_rf = f1_score(y_test, pred_rf)

print("Random Forest Oficial")
print("-" * 40)

print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1-Score : {f1_rf:.4f}")

Random Forest Oficial
----------------------------------------
Accuracy : 0.9755
Precision: 0.1320
Recall   : 0.2391
F1-Score : 0.1701


In [9]:
# Matriz de confusión del modelo Random Forest oficial

from sklearn.metrics import confusion_matrix

cm_rf = confusion_matrix(y_test, pred_rf)

print(cm_rf)

[[12809   217]
 [  105    33]]


In [10]:
# ============================================================
# Balanced Random Forest
# ============================================================

brf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            BalancedRandomForestClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

print("Pipeline Balanced RF creado.")

Pipeline Balanced RF creado.


In [11]:
# ============================================================
# Entrenamiento y evaluación Balanced RF
# ============================================================

brf.fit(X_train, y_train)

pred_brf = brf.predict(X_test)

accuracy_brf = accuracy_score(y_test, pred_brf)
precision_brf = precision_score(y_test, pred_brf)
recall_brf = recall_score(y_test, pred_brf)
f1_brf = f1_score(y_test, pred_brf)

print("Balanced Random Forest")
print("-" * 40)

print(f"Accuracy : {accuracy_brf:.4f}")
print(f"Precision: {precision_brf:.4f}")
print(f"Recall   : {recall_brf:.4f}")
print(f"F1-Score : {f1_brf:.4f}")

Balanced Random Forest
----------------------------------------
Accuracy : 0.8870
Precision: 0.0679
Recall   : 0.7681
F1-Score : 0.1247


In [12]:
# ============================================================
# Matriz de confusión - Balanced RF
# ============================================================

from sklearn.metrics import confusion_matrix

cm_brf = confusion_matrix(y_test, pred_brf)

print(cm_brf)

[[11570  1456]
 [   32   106]]


**Conclusión preliminar Balanced Random Forest**

Balanced Random Forest incrementó fuertemente la detección de siniestros fatales (Recall), pero a costa de un aumento muy significativo de falsos positivos. El F1-Score resultó inferior al obtenido por el Random Forest oficial, por lo que no se considera una mejora bajo el criterio principal definido para el proyecto.

In [13]:
# ============================================================
# SMOTE + Random Forest
# ============================================================

smote_rf = ImbPipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=None,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

print("Pipeline SMOTE + RF creado.")

Pipeline SMOTE + RF creado.


In [14]:
# ============================================================
# Entrenamiento y evaluación SMOTE + RF
# ============================================================

smote_rf.fit(X_train, y_train)

pred_smote_rf = smote_rf.predict(X_test)

accuracy_smote_rf = accuracy_score(y_test, pred_smote_rf)
precision_smote_rf = precision_score(y_test, pred_smote_rf)
recall_smote_rf = recall_score(y_test, pred_smote_rf)
f1_smote_rf = f1_score(y_test, pred_smote_rf)

print("SMOTE + Random Forest")
print("-" * 40)

print(f"Accuracy : {accuracy_smote_rf:.4f}")
print(f"Precision: {precision_smote_rf:.4f}")
print(f"Recall   : {recall_smote_rf:.4f}")
print(f"F1-Score : {f1_smote_rf:.4f}")

SMOTE + Random Forest
----------------------------------------
Accuracy : 0.9893
Precision: 0.4000
Recall   : 0.0435
F1-Score : 0.0784


In [15]:
# ============================================================
# Matriz de confusión - SMOTE + RF
# ============================================================

from sklearn.metrics import confusion_matrix

cm_smote_rf = confusion_matrix(y_test, pred_smote_rf)

print(cm_smote_rf)


[[13017     9]
 [  132     6]]


**Conclusión preliminar – SMOTE + Random Forest**

La incorporación de SMOTE produjo un comportamiento opuesto al observado con Balanced Random Forest.

Mientras que Balanced RF incrementó significativamente el Recall a costa de una gran cantidad de falsos positivos, SMOTE + Random Forest se mostró extremadamente conservador, generando muy pocos falsos positivos pero detectando una proporción mínima de los siniestros fatales.

El modelo obtuvo la mayor Precision de los experimentos realizados hasta el momento (0.40), pero presentó un Recall muy bajo (0.0435), dejando sin detectar la gran mayoría de los casos fatales.

Como consecuencia, el F1-Score (0.0784) resultó considerablemente inferior al obtenido por el Random Forest oficial (0.1701).

Bajo el criterio principal definido para el proyecto —maximizar el equilibrio entre Precision y Recall mediante F1-Score—, SMOTE + Random Forest no representa una mejora respecto del modelo oficial.

In [16]:
# ============================================================
# Pipeline XGBoost
# ============================================================

positivos = y_train.sum()
negativos = len(y_train) - positivos

scale_pos_weight = negativos / positivos

print(f"scale_pos_weight: {scale_pos_weight:.2f}")

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                scale_pos_weight=scale_pos_weight,
                random_state=RANDOM_STATE,
                eval_metric="logloss"
            )
        )
    ]
)

print("Pipeline XGBoost creado.")

scale_pos_weight: 94.56
Pipeline XGBoost creado.


**Observación***

A diferencia de los modelos anteriores, XGBoost permite incorporar el desbalance de clases mediante el parámetro `scale_pos_weight`, evitando generar observaciones sintéticas (SMOTE) o modificar la composición de cada árbol (Balanced RF).

El valor calculado automáticamente (94.56) refleja la proporción observada entre casos no fatales y fatales en el conjunto de entrenamiento.

In [17]:
# ============================================================
# Entrenamiento y evaluación XGBoost
# ============================================================

xgb_model.fit(X_train, y_train)

pred_xgb = xgb_model.predict(X_test)

accuracy_xgb = accuracy_score(y_test, pred_xgb)
precision_xgb = precision_score(y_test, pred_xgb)
recall_xgb = recall_score(y_test, pred_xgb)
f1_xgb = f1_score(y_test, pred_xgb)

print("XGBoost")
print("-" * 40)

print(f"Accuracy : {accuracy_xgb:.4f}")
print(f"Precision: {precision_xgb:.4f}")
print(f"Recall   : {recall_xgb:.4f}")
print(f"F1-Score : {f1_xgb:.4f}")

XGBoost
----------------------------------------
Accuracy : 0.9371
Precision: 0.0951
Recall   : 0.5870
F1-Score : 0.1636


**Conclusión preliminar – XGBoost**

XGBoost mostró el comportamiento más competitivo entre los modelos alternativos evaluados durante la Fase 4B.

La incorporación del parámetro `scale_pos_weight` permitió considerar explícitamente el fuerte desbalance existente en el dataset sin recurrir a técnicas de sobremuestreo o submuestreo. Como resultado, el modelo logró incrementar significativamente la detección de siniestros fatales respecto del Random Forest oficial, alcanzando un Recall de 0.5870.

Sin embargo, esta mejora en la capacidad de detección estuvo acompañada por una disminución de la Precision, lo que produjo un F1-Score de 0.1636, ligeramente inferior al obtenido por el Random Forest oficial (0.1701).

Si bien XGBoost se posicionó como la alternativa más cercana al modelo de referencia, la diferencia observada no resulta suficiente para justificar un reemplazo del modelo oficial bajo los criterios definidos para el proyecto. Los resultados sugieren que futuras mejoras podrían provenir con mayor probabilidad del Feature Engineering derivado de los hallazgos del EDA que de un cambio de algoritmo.

**Comparación de algoritmos**

| Modelo      |  Precision |     Recall |         F1 |
| ----------- | ---------: | ---------: | ---------: |
| RF Oficial  | **0.1320** |     0.2391 | **0.1701** |
| Balanced RF |     0.0679 | **0.7681** |     0.1247 |
| SMOTE + RF  | **0.4000** |     0.0435 |     0.0784 |
| XGBoost     |     0.0951 |     0.5870 |     0.1636 |


Se evaluaron tres enfoques alternativos para abordar el fuerte desbalance presente en el dataset: Balanced Random Forest, SMOTE + Random Forest y XGBoost.

Ninguno de los modelos logró superar el F1-Score obtenido por el Random Forest oficial utilizado en la Fase 4.

Balanced Random Forest incrementó significativamente el Recall, pero generó una cantidad excesiva de falsos positivos. SMOTE + Random Forest obtuvo la mayor Precision, aunque detectó una proporción muy reducida de los siniestros fatales.

XGBoost fue el modelo que mostró el comportamiento más competitivo, alcanzando un F1-Score cercano al del Random Forest oficial y mejorando considerablemente el Recall. Sin embargo, la mejora no fue suficiente para justificar el reemplazo del modelo oficial bajo los criterios definidos para el proyecto.

Los resultados sugieren que el principal potencial de mejora no se encuentra en el cambio de algoritmo, sino posiblemente en la incorporación de nuevas variables derivadas del conocimiento obtenido durante el EDA.

In [18]:
# ============================================================
# Normalización temporal para análisis exploratorio
# ============================================================

horas = pd.to_numeric(
    df["rango_horario"],
    errors="coerce"
)

print(horas.describe())

print("\nValores únicos:")
print(sorted(horas.dropna().unique()))

count    65737.000000
mean        13.642135
std          5.542656
min          0.000000
25%         10.000000
50%         14.000000
75%         18.000000
max         23.000000
Name: rango_horario, dtype: float64

Valores únicos:
[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0]


In [19]:
# ============================================================
# Tasa de fatalidad por hora
# ============================================================

tabla_horaria = pd.crosstab(
    horas,
    df["siniestro_fatal"],
    normalize="index"
)

tabla_horaria["tasa_fatalidad"] = tabla_horaria[1]

tabla_horaria[["tasa_fatalidad"]]

siniestro_fatal,tasa_fatalidad
rango_horario,
0.0,0.009995
1.0,0.017951
2.0,0.018759
3.0,0.057087
4.0,0.050661
5.0,0.041114
6.0,0.024298
7.0,0.020465
8.0,0.009122


In [20]:
# ============================================================
# Tasa de fatalidad por día de semana
# ============================================================

pd.crosstab(
    df["dia_semana"],
    df["siniestro_fatal"],
    normalize="index"
).round(4)

siniestro_fatal,0,1
dia_semana,,
Friday,0.9917,0.0083
Monday,0.9903,0.0097
Saturday,0.9868,0.0132
Sunday,0.9863,0.0137
Thursday,0.9885,0.0115
Tuesday,0.9896,0.0104
Wednesday,0.9913,0.0087


**Justificación de las variables derivadas**

A partir de los hallazgos obtenidos durante el EDA y de un análisis específico realizado en esta fase experimental, se observó que la probabilidad de ocurrencia de siniestros fatales presenta patrones diferenciados tanto por hora del día como por tipo de día.

En particular, las horas comprendidas entre las 03:00 y las 05:00 mostraron una tasa de fatalidad considerablemente superior al resto del día, justificando su agrupación en una categoría específica denominada "Madrugada".

Asimismo, los días sábado y domingo presentaron tasas de fatalidad superiores a las observadas durante los días hábiles, respaldando la creación de la variable binaria "tipo_dia".

Estas transformaciones buscan reducir la fragmentación introducida por el One Hot Encoding y mejorar la capacidad del modelo para capturar patrones temporales observados previamente en el EDA.

In [21]:
# ============================================================
# Feature Engineering
# ============================================================

df_fe = df.copy()

# ------------------------------------------------------------
# Franja horaria
# ------------------------------------------------------------

horas = pd.to_numeric(
    df_fe["rango_horario"],
    errors="coerce"
)

def clasificar_franja(hora):

    if pd.isna(hora):
        return "SIN_DATO"

    elif hora <= 2:
        return "TRASNOCHE"

    elif hora <= 5:
        return "MADRUGADA"

    elif hora <= 11:
        return "MANANA"

    elif hora <= 14:
        return "MEDIODIA"

    elif hora <= 18:
        return "TARDE"

    else:
        return "NOCHE"

df_fe["franja_horaria"] = horas.apply(clasificar_franja)

# ------------------------------------------------------------
# Tipo de día
# ------------------------------------------------------------

df_fe["tipo_dia"] = np.where(
    df_fe["dia_semana"].isin(["Saturday", "Sunday"]),
    "FIN_SEMANA",
    "HABIL"
)

print(df_fe[["franja_horaria", "tipo_dia"]].head())

  franja_horaria tipo_dia
0      TRASNOCHE    HABIL
1          TARDE    HABIL
2       MEDIODIA    HABIL
3         MANANA    HABIL
4       MEDIODIA    HABIL


In [22]:
df_fe["tipo_dia"].value_counts()

tipo_dia
HABIL         52178
FIN_SEMANA    13640
Name: count, dtype: int64

In [23]:
df_fe["franja_horaria"].value_counts()

franja_horaria
TARDE        18140
MANANA       16087
NOCHE        13498
MEDIODIA     12755
TRASNOCHE     3541
MADRUGADA     1716
SIN_DATO        81
Name: count, dtype: int64

In [24]:
# ============================================================
# Dataset experimental con Feature Engineering
# ============================================================

X_fe = df_fe.drop(
    columns=[
        "id_siniestro",
        "fecha_siniestro",
        "siniestro_fatal",
        "dia_semana",
        "rango_horario"
    ]
).copy()

y_fe = df_fe["siniestro_fatal"]

# Corrección de tipos
X_fe["comuna_siniestro"] = X_fe["comuna_siniestro"].astype(str)

print("Shape X_fe:", X_fe.shape)
print("Shape y_fe:", y_fe.shape)

print("\nColumnas:")
print(X_fe.columns.tolist())

Shape X_fe: (65818, 14)
Shape y_fe: (65818,)

Columnas:
['anio_siniestro', 'mes_siniestro', 'dia_siniestro', 'comuna_siniestro', 'tipo_de_via_siniestro', 'participantes_siniestro', 'modo_desplazamiento_victima', 'contraparte_siniestro', 'latitud_siniestro', 'longitud_siniestro', 'edad_promedio_victimas', 'sexo_victima_agregado', 'franja_horaria', 'tipo_dia']


In [25]:
# ============================================================
# División entrenamiento / prueba
# ============================================================

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe,
    y_fe,
    test_size=0.20,
    stratify=y_fe,
    random_state=RANDOM_STATE
)

print("Train:", X_train_fe.shape)
print("Test :", X_test_fe.shape)

Train: (52654, 14)
Test : (13164, 14)


In [26]:
# ============================================================
# Variables numéricas y categóricas
# ============================================================

numericas_fe = [
    "anio_siniestro",
    "mes_siniestro",
    "dia_siniestro",
    "latitud_siniestro",
    "longitud_siniestro",
    "edad_promedio_victimas"
]

categoricas_fe = [
    col for col in X_fe.columns
    if col not in numericas_fe
]

print("Numéricas:", len(numericas_fe))
print("Categóricas:", len(categoricas_fe))

print("\nVariables categóricas:")
print(categoricas_fe)

Numéricas: 6
Categóricas: 8

Variables categóricas:
['comuna_siniestro', 'tipo_de_via_siniestro', 'participantes_siniestro', 'modo_desplazamiento_victima', 'contraparte_siniestro', 'sexo_victima_agregado', 'franja_horaria', 'tipo_dia']


In [27]:
# ============================================================
# Preprocesamiento Feature Engineering
# ============================================================

transformador_numerico_fe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

transformador_categorico_fe = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="SIN_DATO"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor_fe = ColumnTransformer(
    transformers=[
        ("num", transformador_numerico_fe, numericas_fe),
        ("cat", transformador_categorico_fe, categoricas_fe)
    ]
)

print("Preprocesador FE construido correctamente.")

Preprocesador FE construido correctamente.


In [28]:
# ============================================================
# Random Forest Oficial + Feature Engineering
# ============================================================

rf_fe = Pipeline(
    steps=[
        ("preprocessor", preprocessor_fe),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=None,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

rf_fe.fit(X_train_fe, y_train_fe)

pred_rf_fe = rf_fe.predict(X_test_fe)

accuracy_rf_fe = accuracy_score(y_test_fe, pred_rf_fe)
precision_rf_fe = precision_score(y_test_fe, pred_rf_fe)
recall_rf_fe = recall_score(y_test_fe, pred_rf_fe)
f1_rf_fe = f1_score(y_test_fe, pred_rf_fe)

print("Random Forest + Feature Engineering")
print("-" * 50)

print(f"Accuracy : {accuracy_rf_fe:.4f}")
print(f"Precision: {precision_rf_fe:.4f}")
print(f"Recall   : {recall_rf_fe:.4f}")
print(f"F1-Score : {f1_rf_fe:.4f}")

Random Forest + Feature Engineering
--------------------------------------------------
Accuracy : 0.9734
Precision: 0.1395
Recall   : 0.2971
F1-Score : 0.1898


**Conclusión preliminar – Random Forest con Feature Engineering**

| Modelo     |  Precision |     Recall |         F1 |
| ---------- | ---------: | ---------: | ---------: |
| RF Oficial |     0.1320 |     0.2391 |     0.1701 |
| RF + FE    | **0.1395** | **0.2971** | **0.1898** |


La sustitución de las variables temporales originales (`dia_semana` y `rango_horario`) por las variables derivadas (`tipo_dia` y `franja_horaria`) produjo una mejora consistente en el desempeño del modelo.

El F1-Score aumentó de 0.1701 a 0.1898, acompañado por incrementos tanto en Precision como en Recall. Este resultado sugiere que la nueva representación temporal permite capturar con mayor eficacia los patrones identificados previamente durante el EDA.

A diferencia de los experimentos basados en cambios de algoritmo, donde las mejoras en una métrica solían implicar deterioros importantes en otra, el Feature Engineering mostró una mejora equilibrada y metodológicamente consistente.

Los resultados respaldan la hipótesis planteada al cierre de la Fase 4: parte de la información temporal relevante se encontraba fragmentada por el One Hot Encoding de las variables originales y puede representarse de forma más efectiva mediante agrupaciones basadas en evidencia observada en los datos.

In [29]:
from sklearn.metrics import confusion_matrix

cm_rf_fe = confusion_matrix(
    y_test_fe,
    pred_rf_fe
)

print(cm_rf_fe)

[[12773   253]
 [   97    41]]


**Observación sobre la matriz de confusión**

La incorporación de las variables `franja_horaria` y `tipo_dia` permitió incrementar la cantidad de siniestros fatales correctamente identificados por el modelo, pasando de 33 a 41 verdaderos positivos.

Esta mejora se obtuvo a costa de un aumento moderado de falsos positivos (de 217 a 253), aunque el balance general resultó favorable, reflejándose en incrementos simultáneos de Precision, Recall y F1-Score.

Los resultados sugieren que las nuevas variables temporales capturan información relevante que no estaba siendo aprovechada eficientemente por las variables originales (`dia_semana` y `rango_horario`), reforzando la hipótesis planteada al cierre de la Fase 4.

No obstante, antes de considerar esta mejora como concluyente, resulta necesario verificar su estabilidad mediante validación sobre múltiples particiones de datos.

In [30]:
# ============================================================
# Validación multisemilla
# RF Oficial vs RF + Feature Engineering
# ============================================================

semillas = [42, 7, 123, 2025, 999]

resultados = []

for semilla in semillas:

    # --------------------------------------------------------
    # Split RF original
    # --------------------------------------------------------

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=semilla
    )

    rf_original = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=None,
                    min_samples_leaf=5,
                    class_weight="balanced",
                    random_state=semilla,
                    n_jobs=-1
                )
            )
        ]
    )

    rf_original.fit(X_train, y_train)

    pred_original = rf_original.predict(X_test)

    f1_original = f1_score(y_test, pred_original)

    # --------------------------------------------------------
    # Split RF + Feature Engineering
    # --------------------------------------------------------

    X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
        X_fe,
        y_fe,
        test_size=0.20,
        stratify=y_fe,
        random_state=semilla
    )

    rf_feature = Pipeline(
        steps=[
            ("preprocessor", preprocessor_fe),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=None,
                    min_samples_leaf=5,
                    class_weight="balanced",
                    random_state=semilla,
                    n_jobs=-1
                )
            )
        ]
    )

    rf_feature.fit(X_train_fe, y_train_fe)

    pred_feature = rf_feature.predict(X_test_fe)

    f1_feature = f1_score(y_test_fe, pred_feature)

    resultados.append(
        {
            "semilla": semilla,
            "f1_original": round(f1_original, 4),
            "f1_feature": round(f1_feature, 4),
            "diferencia": round(f1_feature - f1_original, 4)
        }
    )

resultados_df = pd.DataFrame(resultados)

resultados_df

,semilla,f1_original,f1_feature,diferencia
0,42,0.1701,0.1898,0.0197
1,7,0.2434,0.2379,-0.0055
2,123,0.2228,0.2419,0.0191
3,2025,0.2537,0.2622,0.0085
4,999,0.2494,0.2634,0.0140



**Conclusión – Validación multisemilla**

| Semilla | RF Original |    RF + FE | Diferencia |
| ------- | ----------: | ---------: | ---------: |
| 42      |      0.1701 | **0.1898** |    +0.0197 |
| 7       |  **0.2434** |     0.2379 |    -0.0055 |
| 123     |      0.2228 | **0.2419** |    +0.0191 |
| 2025    |      0.2537 | **0.2622** |    +0.0085 |
| 999     |      0.2494 | **0.2634** |    +0.0140 |


Con el objetivo de verificar que la mejora observada no dependiera de una única partición de datos, se repitió el entrenamiento utilizando las mismas semillas empleadas durante la Fase 4.

El modelo con Feature Engineering obtuvo un F1-Score superior al modelo original en cuatro de las cinco semillas evaluadas. La única excepción correspondió a la semilla 7, donde la diferencia observada fue reducida.

En promedio, el modelo con variables derivadas alcanzó un F1-Score aproximado de 0.239 frente a 0.228 del modelo original, evidenciando una mejora consistente y estable (aproximadamente un 4,9% de mejora).

Estos resultados respaldan la hipótesis de que la agrupación de variables temporales basada en los hallazgos del EDA permite representar mejor la información disponible y mejorar el desempeño predictivo del modelo sin necesidad de recurrir a algoritmos más complejos.

In [31]:
# ============================================================
# Entrenamiento final RF + FE
# ============================================================

rf_fe.fit(X_train_fe, y_train_fe)

# ============================================================
# Nombres de variables transformadas
# ============================================================

feature_names = rf_fe.named_steps[
    "preprocessor"
].get_feature_names_out()

importancias = rf_fe.named_steps[
    "model"
].feature_importances_

feature_importance_df = pd.DataFrame(
    {
        "variable": feature_names,
        "importancia": importancias
    }
)

feature_importance_df = (
    feature_importance_df
    .sort_values(
        by="importancia",
        ascending=False
    )
)

feature_importance_df.head(20)

,variable,importancia
5,num__edad_promedio_victimas,0.074362
43,cat__tipo_de_via_siniestro_SIN_DATO,0.066275
248,cat__sexo_victima_agregado_SD,0.058119
3,num__latitud_siniestro,0.052066
4,num__longitud_siniestro,0.045892
0,num__anio_siniestro,0.040619
2,num__dia_siniestro,0.038213
240,cat__contraparte_siniestro_SD,0.035725
224,cat__modo_desplazamiento_victima_PEATON,0.035401
1,num__mes_siniestro,0.032124


franja_horaria_MADRUGADA aparece dentro del Top 20,
aportando evidencia adicional a favor del Feature Engineering temporal.

In [32]:
# ============================================================
# Comparación de algoritmos - Fase 4B
# ============================================================

comparacion_algoritmos = pd.DataFrame(
    {
        "Modelo": [
            "RF Oficial",
            "Balanced RF",
            "SMOTE + RF",
            "XGBoost"
        ],
        "Accuracy": [
            0.9755,
            0.8870,
            0.9893,
            0.9371
        ],
        "Precision": [
            0.1320,
            0.0679,
            0.4000,
            0.0951
        ],
        "Recall": [
            0.2391,
            0.7681,
            0.0435,
            0.5870
        ],
        "F1": [
            0.1701,
            0.1247,
            0.0784,
            0.1636
        ]
    }
)

comparacion_algoritmos

,Modelo,Accuracy,Precision,Recall,F1
0,RF Oficial,0.9755,0.1320,0.2391,0.1701
1,Balanced RF,0.8870,0.0679,0.7681,0.1247
2,SMOTE + RF,0.9893,0.4000,0.0435,0.0784
3,XGBoost,0.9371,0.0951,0.5870,0.1636


In [33]:
# ============================================================
# Impacto del Feature Engineering
# ============================================================

comparacion_feature_engineering = pd.DataFrame(
    {
        "Modelo": [
            "RF Oficial",
            "RF + Feature Engineering"
        ],
        "Precision": [
            0.1320,
            0.1395
        ],
        "Recall": [
            0.2391,
            0.2971
        ],
        "F1": [
            0.1701,
            0.1898
        ]
    }
)

comparacion_feature_engineering["Delta F1"] = [
    0,
    round(0.1898 - 0.1701, 4)
]

comparacion_feature_engineering

,Modelo,Precision,Recall,F1,Delta F1
0,RF Oficial,0.1320,0.2391,0.1701,0.0000
1,RF + Feature Engineering,0.1395,0.2971,0.1898,0.0197


In [34]:
# ============================================================
# Validación multisemilla
# ============================================================

comparacion_semillas = pd.DataFrame(
    {
        "Semilla": [42, 7, 123, 2025, 999],
        "RF Original": [
            0.1701,
            0.2434,
            0.2228,
            0.2537,
            0.2494
        ],
        "RF + FE": [
            0.1898,
            0.2379,
            0.2419,
            0.2622,
            0.2634
        ]
    }
)

comparacion_semillas["Diferencia"] = (
    comparacion_semillas["RF + FE"]
    - comparacion_semillas["RF Original"]
)

promedio_original = comparacion_semillas["RF Original"].mean()
promedio_fe = comparacion_semillas["RF + FE"].mean()

fila_promedio = pd.DataFrame(
    {
        "Semilla": ["PROMEDIO"],
        "RF Original": [round(promedio_original, 4)],
        "RF + FE": [round(promedio_fe, 4)],
        "Diferencia": [round(promedio_fe - promedio_original, 4)]
    }
)

comparacion_semillas = pd.concat(
    [comparacion_semillas, fila_promedio],
    ignore_index=True
)

comparacion_semillas

,Semilla,RF Original,RF + FE,Diferencia
0,42,0.1701,0.1898,0.0197
1,7,0.2434,0.2379,-0.0055
2,123,0.2228,0.2419,0.0191
3,2025,0.2537,0.2622,0.0085
4,999,0.2494,0.2634,0.0140
5,PROMEDIO,0.2279,0.2390,0.0112


In [35]:
# ============================================================
# Top 10 variables más importantes
# ============================================================

top10_variables = (
    feature_importance_df
    .head(10)
    .copy()
)

top10_variables["importancia"] = (
    top10_variables["importancia"]
    .round(4)
)

top10_variables

,variable,importancia
5,num__edad_promedio_victimas,0.0744
43,cat__tipo_de_via_siniestro_SIN_DATO,0.0663
248,cat__sexo_victima_agregado_SD,0.0581
3,num__latitud_siniestro,0.0521
4,num__longitud_siniestro,0.0459
0,num__anio_siniestro,0.0406
2,num__dia_siniestro,0.0382
240,cat__contraparte_siniestro_SD,0.0357
224,cat__modo_desplazamiento_victima_PEATON,0.0354
1,num__mes_siniestro,0.0321


**Conclusiones**

Cuadro 1 – Comparación de algoritmos

| Modelo      |   Accuracy |  Precision |     Recall |         F1 |
| ----------- | ---------: | ---------: | ---------: | ---------: |
| RF Oficial  |     0.9755 |     0.1320 |     0.2391 | **0.1701** |
| Balanced RF |     0.8870 |     0.0679 | **0.7681** |     0.1247 |
| SMOTE + RF  | **0.9893** | **0.4000** |     0.0435 |     0.0784 |
| XGBoost     |     0.9371 |     0.0951 |     0.5870 |     0.1636 |


Cuadro 2 – Impacto del Feature Engineering

| Modelo                   |  Precision |     Recall |         F1 |
| ------------------------ | ---------: | ---------: | ---------: |
| RF Oficial               |     0.1320 |     0.2391 |     0.1701 |
| RF + Feature Engineering | **0.1395** | **0.2971** | **0.1898** |
| Diferencia               |    +0.0075 |    +0.0580 |    +0.0197 |


Cuadro 3 – Validación multisemilla

| Semilla      | RF Original |    RF + FE |  Diferencia |
| ------------ | ----------: | ---------: | ----------: |
| 42           |      0.1701 | **0.1898** |     +0.0197 |
| 7            |  **0.2434** |     0.2379 |     -0.0055 |
| 123          |      0.2228 | **0.2419** |     +0.0191 |
| 2025         |      0.2537 | **0.2622** |     +0.0085 |
| 999          |      0.2494 | **0.2634** |     +0.0140 |
| **Promedio** |  **0.2279** | **0.2390** | **+0.0111** |


La principal mejora identificada no proviene de la incorporación de algoritmos más complejos, sino de una representación más adecuada de la información temporal.

La sustitución de las variables originales dia_semana y rango_horario por las variables derivadas tipo_dia y franja_horaria produjo una mejora consistente, estable e interpretable del desempeño del modelo.

Por lo tanto, se recomienda mantener el Random Forest como algoritmo principal e incorporar las nuevas variables temporales como mejora metodológicamente respaldada.

**Exportación de resultados para base de datos**

Se exportan los resultados principales de la fase experimental para que puedan ser utilizados posteriormente en la construcción de la base SQLite del proyecto.

Esta exportación permite conservar la trazabilidad de los modelos avanzados evaluados, la comparación con el modelo oficial y la validación multisemilla.

In [37]:
# Rutas de exportación para la Fase 4B
ruta_resultados_fase4b = "../data_processed/resultados_modelos_fase4b.csv"
ruta_predicciones_final = "../data_processed/predicciones_modelo_final.csv"

# ------------------------------------------------------------
# Resultados principales de modelos avanzados
# ------------------------------------------------------------

resultados_fase4b = pd.DataFrame([
    {
        "fase": "Fase 4B",
        "experimento": "Modelos avanzados",
        "modelo": "Balanced RF",
        "semilla": RANDOM_STATE,
        "accuracy": accuracy_brf,
        "precision": precision_brf,
        "recall": recall_brf,
        "f1_score": f1_brf,
        "es_modelo_final": 0
    },
    {
        "fase": "Fase 4B",
        "experimento": "Modelos avanzados",
        "modelo": "SMOTE + Random Forest",
        "semilla": RANDOM_STATE,
        "accuracy": accuracy_smote_rf,
        "precision": precision_smote_rf,
        "recall": recall_smote_rf,
        "f1_score": f1_smote_rf,
        "es_modelo_final": 0
    },
    {
        "fase": "Fase 4B",
        "experimento": "Modelos avanzados",
        "modelo": "XGBoost",
        "semilla": RANDOM_STATE,
        "accuracy": accuracy_xgb,
        "precision": precision_xgb,
        "recall": recall_xgb,
        "f1_score": f1_xgb,
        "es_modelo_final": 0
    },
    {
        "fase": "Fase 4B",
        "experimento": "Feature Engineering temporal",
        "modelo": "Random Forest + Feature Engineering temporal",
        "semilla": RANDOM_STATE,
        "accuracy": accuracy_rf_fe,
        "precision": precision_rf_fe,
        "recall": recall_rf_fe,
        "f1_score": f1_rf_fe,
        "es_modelo_final": 1
    }
])

# ------------------------------------------------------------
# Validación multisemilla en formato largo
# ------------------------------------------------------------

resultados_semillas_original = resultados_df[["semilla", "f1_original"]].copy()
resultados_semillas_original["fase"] = "Fase 4B"
resultados_semillas_original["experimento"] = "Validación multisemilla"
resultados_semillas_original["modelo"] = "Random Forest oficial"
resultados_semillas_original["accuracy"] = None
resultados_semillas_original["precision"] = None
resultados_semillas_original["recall"] = None
resultados_semillas_original["f1_score"] = resultados_semillas_original["f1_original"]
resultados_semillas_original["es_modelo_final"] = 0

resultados_semillas_fe = resultados_df[["semilla", "f1_feature"]].copy()
resultados_semillas_fe["fase"] = "Fase 4B"
resultados_semillas_fe["experimento"] = "Validación multisemilla"
resultados_semillas_fe["modelo"] = "Random Forest + Feature Engineering temporal"
resultados_semillas_fe["accuracy"] = None
resultados_semillas_fe["precision"] = None
resultados_semillas_fe["recall"] = None
resultados_semillas_fe["f1_score"] = resultados_semillas_fe["f1_feature"]
resultados_semillas_fe["es_modelo_final"] = 1

# Dejamos ambas validaciones con el mismo esquema de columnas
columnas_resultados = [
    "fase",
    "experimento",
    "modelo",
    "semilla",
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "es_modelo_final"
]

resultados_semillas_original = resultados_semillas_original[columnas_resultados]
resultados_semillas_fe = resultados_semillas_fe[columnas_resultados]
resultados_fase4b = resultados_fase4b[columnas_resultados]

# Unificamos resultados principales y validación multisemilla
resultados_fase4b_exportar = pd.concat(
    [
        resultados_fase4b,
        resultados_semillas_original,
        resultados_semillas_fe
    ],
    ignore_index=True
)

# Exportamos resultados de la Fase 4B
resultados_fase4b_exportar.to_csv(
    ruta_resultados_fase4b,
    index=False
)

# ------------------------------------------------------------
# Predicciones del modelo final
# ------------------------------------------------------------

predicciones_modelo_final = pd.DataFrame({
    "id_siniestro": df.loc[X_test_fe.index, "id_siniestro"],
    "valor_real": y_test_fe,
    "valor_predicho": pred_rf_fe
})

# Agregamos la probabilidad estimada de que el siniestro sea fatal
predicciones_modelo_final["probabilidad_fatal"] = rf_fe.predict_proba(X_test_fe)[:, 1]

# Exportamos predicciones del modelo final
predicciones_modelo_final.to_csv(
    ruta_predicciones_final,
    index=False
)

print("Archivos exportados correctamente:")
print(ruta_resultados_fase4b)
print(ruta_predicciones_final)

print("\nDimensiones exportadas:")
print("Resultados Fase 4B:", resultados_fase4b_exportar.shape)
print("Predicciones modelo final:", predicciones_modelo_final.shape)

Archivos exportados correctamente:
../data_processed/resultados_modelos_fase4b.csv
../data_processed/predicciones_modelo_final.csv

Dimensiones exportadas:
Resultados Fase 4B: (14, 9)
Predicciones modelo final: (13164, 4)
